# Task 2 — Model Building and Training

**What this notebook does:**
1. Loads the model-ready splits saved by `feature-engineering.ipynb`
2. Trains a **Logistic Regression** baseline on both datasets
3. Trains an **XGBoost** ensemble model on both datasets
4. Evaluates all models using **AUC-PR**, **F1-Score**, and **Confusion Matrix**
5. Performs **Stratified K-Fold cross-validation** (k=5) for reliable estimates
6. Compares all models side-by-side and selects the best one
7. Saves the best models to `models/`

**Why these metrics?**
- **AUC-PR**: Area Under Precision-Recall Curve — the standard metric for imbalanced  
  classification. Unlike AUC-ROC, it is not inflated by the large number of true negatives.
- **F1-Score**: Harmonic mean of Precision and Recall — balances catching fraud (recall)  
  against not over-flagging legitimate transactions (precision).
- **Confusion Matrix**: Shows the raw counts of TP, TN, FP, FN — directly interpretable  
  by business stakeholders.
- ❌ **Accuracy**: NOT used — a model predicting "always legitimate" scores 99.83% on  
  the credit card dataset while catching zero fraud.

**Prerequisites:** Run `eda-fraud-data.ipynb` → `eda-creditcard.ipynb` → `feature-engineering.ipynb`  
so that `data/processed/` contains all six CSV splits and two scaler `.pkl` files.

## 1. Setup & Imports

In [ ]:
import os, sys, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

warnings.filterwarnings('ignore')

# ── Scikit-learn ──────────────────────────────────────────────────────────────
from sklearn.linear_model   import LogisticRegression
from sklearn.metrics        import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_recall_curve,
    auc,
    average_precision_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate

# ── XGBoost ───────────────────────────────────────────────────────────────────
from xgboost import XGBClassifier

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (10, 5)

DATA_PROCESSED = os.path.join('..', 'data', 'processed')
MODELS_DIR     = os.path.join('..', 'models')
FIGURES_DIR    = 'figures'

os.makedirs(MODELS_DIR,   exist_ok=True)
os.makedirs(FIGURES_DIR,  exist_ok=True)
print('✓ Setup complete')

## 2. Shared Evaluation Helpers

In [ ]:
def evaluate_model(model, X_test, y_test, model_name, dataset_name, ax_cm=None):
    """
    Compute and display evaluation metrics for a fitted model.

    Returns a dict with keys: model_name, dataset, auc_pr, f1, precision, recall.

    Parameters
    ----------
    model       : fitted classifier with predict() and predict_proba()
    X_test      : test features
    y_test      : true labels
    model_name  : string label for display
    dataset_name: 'E-commerce' or 'Credit Card'
    ax_cm       : optional matplotlib Axes to draw the confusion matrix on
    """
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]   # probability of class 1 (fraud)

    # ── AUC-PR ────────────────────────────────────────────────────────────────
    # average_precision_score is equivalent to the area under the PR curve
    # and is more numerically stable than computing precision_recall_curve then auc()
    auc_pr = average_precision_score(y_test, y_proba)

    # ── F1-Score ──────────────────────────────────────────────────────────────
    # zero_division=0: avoids divide-by-zero warning when a class has no predictions
    f1 = f1_score(y_test, y_pred, zero_division=0)

    # ── Precision-Recall Curve ────────────────────────────────────────────────
    precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_proba)

    # ── Confusion Matrix ──────────────────────────────────────────────────────
    cm = confusion_matrix(y_test, y_pred)

    # ── Print summary ─────────────────────────────────────────────────────────
    print(f"\n{'='*55}")
    print(f"  {model_name} — {dataset_name}")
    print(f"{'='*55}")
    print(f"  AUC-PR  : {auc_pr:.4f}")
    print(f"  F1-Score: {f1:.4f}")
    print()
    print(classification_report(y_test, y_pred,
          target_names=['Legitimate', 'Fraud'], zero_division=0))

    # ── Plot confusion matrix on provided axes (if given) ─────────────────────
    if ax_cm is not None:
        disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                      display_labels=['Legit', 'Fraud'])
        disp.plot(ax=ax_cm, colorbar=False, cmap='Blues')
        ax_cm.set_title(f"{model_name}\n{dataset_name}", fontsize=10)

    return {
        'model'    : model_name,
        'dataset'  : dataset_name,
        'auc_pr'   : round(auc_pr, 4),
        'f1'       : round(f1, 4),
        'precision': round(float(cm[1,1] / (cm[0,1] + cm[1,1]) if (cm[0,1]+cm[1,1])>0 else 0), 4),
        'recall'   : round(float(cm[1,1] / (cm[1,0] + cm[1,1]) if (cm[1,0]+cm[1,1])>0 else 0), 4),
        'pr_curve' : (precision_vals, recall_vals),
    }


def plot_pr_curves(results_list, title, ax):
    """
    Plot Precision-Recall curves for multiple models on the same axes.

    A random baseline (horizontal line at fraud rate) is also drawn.
    The area under each curve (AUC-PR) is shown in the legend.

    Parameters
    ----------
    results_list : list of dicts returned by evaluate_model()
    title        : axes title string
    ax           : matplotlib Axes
    """
    colors = ['steelblue', 'tomato', 'mediumseagreen', 'darkorange']
    for i, res in enumerate(results_list):
        p, r = res['pr_curve']
        ax.plot(r, p, color=colors[i % len(colors)], linewidth=2,
                label=f"{res['model']}  (AUC-PR = {res['auc_pr']:.4f})")
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend(loc='upper right', fontsize=9)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1.05])


def run_cross_validation(model, X_train, y_train, model_name, dataset_name, cv=5):
    """
    Run Stratified K-Fold cross-validation and print mean ± std of AUC-PR and F1.

    Why stratified?
        StratifiedKFold preserves the fraud rate in every fold.
        Without stratification, some folds might have zero fraud examples,
        which would produce misleading (and often zero) AUC-PR scores.

    Parameters
    ----------
    model       : unfitted classifier
    X_train     : resampled training features (after SMOTE)
    y_train     : resampled training labels
    model_name  : string label
    dataset_name: string label
    cv          : number of folds (default 5)
    """
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)

    # cross_validate computes multiple metrics in one call
    # 'average_precision' = AUC-PR
    scores = cross_validate(
        model, X_train, y_train,
        cv=skf,
        scoring={
            'auc_pr': 'average_precision',
            'f1'    : 'f1',
        },
        return_train_score=False,
        n_jobs=-1,   # use all CPU cores
    )

    print(f"\n--- {model_name} CV ({cv}-fold) — {dataset_name} ---")
    print(f"  AUC-PR : {scores['test_auc_pr'].mean():.4f}  ±  {scores['test_auc_pr'].std():.4f}")
    print(f"  F1     : {scores['test_f1'].mean():.4f}  ±  {scores['test_f1'].std():.4f}")

    return {
        'auc_pr_mean': round(scores['test_auc_pr'].mean(), 4),
        'auc_pr_std' : round(scores['test_auc_pr'].std(),  4),
        'f1_mean'    : round(scores['test_f1'].mean(), 4),
        'f1_std'     : round(scores['test_f1'].std(),  4),
    }

print('✓ Helper functions defined')

---
## PART A — E-commerce Fraud Data (`Fraud_Data.csv`)
---

## 3. Load E-commerce Splits

In [ ]:
# Load the SMOTE-balanced training set and the real-world test set
# These were saved by feature-engineering.ipynb

X_train_f = pd.read_csv(os.path.join(DATA_PROCESSED, 'fraud_X_train.csv'))
y_train_f = pd.read_csv(os.path.join(DATA_PROCESSED, 'fraud_y_train.csv')).squeeze()
X_test_f  = pd.read_csv(os.path.join(DATA_PROCESSED, 'fraud_X_test.csv'))
y_test_f  = pd.read_csv(os.path.join(DATA_PROCESSED, 'fraud_y_test.csv')).squeeze()

# .squeeze() converts a single-column DataFrame into a Series
# (sklearn expects a 1-D array for y, not a DataFrame)

print("E-commerce splits loaded:")
print(f"  X_train : {X_train_f.shape}  |  fraud rate: {y_train_f.mean()*100:.2f}%  (SMOTE-balanced)")
print(f"  X_test  : {X_test_f.shape}   |  fraud rate: {y_test_f.mean()*100:.2f}%  (real-world)")

## 4. Baseline Model — Logistic Regression (E-commerce)

**Why Logistic Regression as a baseline?**
- It is the simplest linear classifier — if it performs well, the problem may not need
  a complex model, which is always preferable in production.
- Its coefficients are directly interpretable: a positive coefficient means the feature
  pushes toward predicting fraud.
- It sets a performance floor: any ensemble model must beat this to justify its added complexity.

**Key parameter: `max_iter=1000`**  
The default (100) often fails to converge on large datasets. Setting 1000 gives the
optimiser enough iterations to find a good solution.

In [ ]:
# C=1.0 is the default regularisation strength (inverse of lambda)
# solver='lbfgs' works well for binary classification on medium-sized datasets
# random_state ensures reproducible results
lr_fraud = LogisticRegression(max_iter=1000, solver='lbfgs',
                               C=1.0, random_state=42)
lr_fraud.fit(X_train_f, y_train_f)
print('✓ Logistic Regression trained on e-commerce data')

### 4.1 Cross-Validation

In [ ]:
# We run CV on the SMOTE-balanced training set.
# CV here estimates how stable the model is across different subsets of training data.
# clone() is not needed because cross_validate handles it internally.
lr_fraud_cv = run_cross_validation(
    LogisticRegression(max_iter=1000, solver='lbfgs', C=1.0, random_state=42),
    X_train_f, y_train_f,
    model_name='Logistic Regression', dataset_name='E-commerce'
)

### 4.2 Test Set Evaluation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

lr_fraud_results = evaluate_model(
    lr_fraud, X_test_f, y_test_f,
    model_name='Logistic Regression',
    dataset_name='E-commerce',
    ax_cm=axes[0]
)

# Precision-Recall curve for this model alone
plot_pr_curves([lr_fraud_results], 'PR Curve — LR (E-commerce)', axes[1])

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'lr_fraud_eval.png'), dpi=150, bbox_inches='tight')
plt.show()

## 5. Ensemble Model — XGBoost (E-commerce)

**Why XGBoost?**
- Consistently outperforms Logistic Regression on tabular data with non-linear relationships.
- Handles feature interactions automatically (no need to manually engineer interaction terms).
- Built-in feature importance scores — useful as a baseline before SHAP in Task 3.
- Widely used in production fraud detection systems.

**Key hyperparameters tuned:**

| Parameter | Value | Reasoning |
|-----------|-------|-----------|
| `n_estimators` | 300 | More trees = more expressive; 300 is a good balance for this dataset size |
| `max_depth` | 6 | Controls tree depth; deeper = more complex = higher overfitting risk |
| `learning_rate` | 0.1 | Step size per boosting round; lower = more conservative but often better |
| `subsample` | 0.8 | Each tree uses 80% of training rows — reduces overfitting |
| `colsample_bytree` | 0.8 | Each tree uses 80% of features — adds regularisation |
| `eval_metric` | 'aucpr' | Tells XGBoost to optimise for AUC-PR internally |
| `use_label_encoder` | False | Suppresses a deprecation warning in recent XGBoost versions |

In [ ]:
xgb_fraud = XGBClassifier(
    n_estimators      = 300,
    max_depth         = 6,
    learning_rate     = 0.1,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    eval_metric       = 'aucpr',
    use_label_encoder = False,
    random_state      = 42,
    n_jobs            = -1,     # use all available CPU cores
)

xgb_fraud.fit(X_train_f, y_train_f)
print('✓ XGBoost trained on e-commerce data')

### 5.1 Cross-Validation

In [ ]:
xgb_fraud_cv = run_cross_validation(
    XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        use_label_encoder=False, eval_metric='aucpr',
        random_state=42, n_jobs=-1
    ),
    X_train_f, y_train_f,
    model_name='XGBoost', dataset_name='E-commerce'
)

### 5.2 Test Set Evaluation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

xgb_fraud_results = evaluate_model(
    xgb_fraud, X_test_f, y_test_f,
    model_name='XGBoost',
    dataset_name='E-commerce',
    ax_cm=axes[0]
)

plot_pr_curves([xgb_fraud_results], 'PR Curve — XGBoost (E-commerce)', axes[1])

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'xgb_fraud_eval.png'), dpi=150, bbox_inches='tight')
plt.show()

## 6. Model Comparison — E-commerce

In [ ]:
# Side-by-side comparison table
fraud_comparison = pd.DataFrame([
    {**{'model': r['model']}, **{k: v for k, v in r.items() if k not in ('model','dataset','pr_curve')}}
    for r in [lr_fraud_results, xgb_fraud_results]
])

print("=== E-commerce Model Comparison (Test Set) ===")
print(fraud_comparison.to_string(index=False))

In [ ]:
# PR curves side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_pr_curves([lr_fraud_results, xgb_fraud_results],
               'Precision-Recall Curves — E-commerce', axes[0])

# Bar chart comparison
metrics = ['auc_pr', 'f1', 'precision', 'recall']
x = np.arange(len(metrics))
width = 0.35

bars1 = axes[1].bar(x - width/2, [lr_fraud_results[m]  for m in metrics],
                    width, label='Logistic Regression', color='steelblue',  edgecolor='white')
bars2 = axes[1].bar(x + width/2, [xgb_fraud_results[m] for m in metrics],
                    width, label='XGBoost',             color='tomato',     edgecolor='white')

axes[1].set_xticks(x)
axes[1].set_xticklabels(['AUC-PR', 'F1', 'Precision', 'Recall'])
axes[1].set_title('Metric Comparison — E-commerce', fontsize=11, fontweight='bold')
axes[1].set_ylim(0, 1.1)
axes[1].legend()

for bar in bars1:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

plt.suptitle('E-commerce Fraud — Model Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fraud_model_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

### 6.1 Feature Importance — XGBoost (E-commerce)

Understanding which features XGBoost relies on most gives us a preview of the SHAP
analysis in Task 3. XGBoost's built-in importance is based on **gain** — how much
each feature improves the model when it is used in a split.

In [ ]:
# get_booster().get_score() returns feature importance as a dict
# importance_type='gain' measures average improvement in accuracy per split using this feature
importance_dict = xgb_fraud.get_booster().get_score(importance_type='gain')

# Some features may have zero importance (never used) — they won't appear in the dict
importance_series = (
    pd.Series(importance_dict)
    .sort_values(ascending=False)
    .head(15)
)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(importance_series.index, importance_series.values,
               color='steelblue', edgecolor='white')
ax.invert_yaxis()
ax.set_xlabel('Feature Importance (Gain)')
ax.set_title('Top 15 Feature Importances — XGBoost (E-commerce)',
             fontsize=12, fontweight='bold')
ax.axvline(importance_series.values.mean(), color='tomato', linestyle='--',
           linewidth=1.5, label='Mean importance')
ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'xgb_fraud_feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()

print("Top 15 features by XGBoost gain (e-commerce):")
print(importance_series.to_string())

---
## PART B — Bank Credit Card Data (`creditcard.csv`)
---

## 7. Load Credit Card Splits

In [ ]:
X_train_cc = pd.read_csv(os.path.join(DATA_PROCESSED, 'cc_X_train.csv'))
y_train_cc = pd.read_csv(os.path.join(DATA_PROCESSED, 'cc_y_train.csv')).squeeze()
X_test_cc  = pd.read_csv(os.path.join(DATA_PROCESSED, 'cc_X_test.csv'))
y_test_cc  = pd.read_csv(os.path.join(DATA_PROCESSED, 'cc_y_test.csv')).squeeze()

print("Credit card splits loaded:")
print(f"  X_train : {X_train_cc.shape}  |  fraud rate: {y_train_cc.mean()*100:.2f}%  (SMOTE-balanced)")
print(f"  X_test  : {X_test_cc.shape}   |  fraud rate: {y_test_cc.mean()*100:.4f}%  (real-world)")

## 8. Baseline Model — Logistic Regression (Credit Card)

Same configuration as the e-commerce baseline for a fair comparison.
The credit card dataset's extreme imbalance (0.17%) means this model
will likely struggle more — which is expected and justifies the ensemble.

In [ ]:
lr_cc = LogisticRegression(max_iter=1000, solver='lbfgs', C=1.0, random_state=42)
lr_cc.fit(X_train_cc, y_train_cc)
print('✓ Logistic Regression trained on credit card data')

### 8.1 Cross-Validation

In [ ]:
lr_cc_cv = run_cross_validation(
    LogisticRegression(max_iter=1000, solver='lbfgs', C=1.0, random_state=42),
    X_train_cc, y_train_cc,
    model_name='Logistic Regression', dataset_name='Credit Card'
)

### 8.2 Test Set Evaluation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

lr_cc_results = evaluate_model(
    lr_cc, X_test_cc, y_test_cc,
    model_name='Logistic Regression',
    dataset_name='Credit Card',
    ax_cm=axes[0]
)

plot_pr_curves([lr_cc_results], 'PR Curve — LR (Credit Card)', axes[1])

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'lr_cc_eval.png'), dpi=150, bbox_inches='tight')
plt.show()

## 9. Ensemble Model — XGBoost (Credit Card)

For the credit card dataset we add `scale_pos_weight` as an additional
safeguard against imbalance **at the model level**. This parameter tells
XGBoost to penalise misclassifying fraud examples more than legitimate ones.

`scale_pos_weight = count(negative) / count(positive)`

Even though SMOTE already balanced the training set, including this parameter
reinforces the model's sensitivity to the minority class.

In [ ]:
# Calculate scale_pos_weight from the ORIGINAL (pre-SMOTE) class distribution
# We use the raw value from the cleaned dataset, not the resampled one
neg_count = (y_train_cc == 0).sum()
pos_count = (y_train_cc == 1).sum()
spw       = neg_count / pos_count  # will be ~1.0 because SMOTE balanced it

print(f"Training set (post-SMOTE) — Negative: {neg_count:,}  Positive: {pos_count:,}")
print(f"scale_pos_weight = {spw:.2f}  (close to 1.0 because training data is already balanced)")

xgb_cc = XGBClassifier(
    n_estimators      = 300,
    max_depth         = 6,
    learning_rate     = 0.1,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    scale_pos_weight  = spw,
    eval_metric       = 'aucpr',
    use_label_encoder = False,
    random_state      = 42,
    n_jobs            = -1,
)

xgb_cc.fit(X_train_cc, y_train_cc)
print('✓ XGBoost trained on credit card data')

### 9.1 Cross-Validation

In [ ]:
xgb_cc_cv = run_cross_validation(
    XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        use_label_encoder=False, eval_metric='aucpr',
        random_state=42, n_jobs=-1
    ),
    X_train_cc, y_train_cc,
    model_name='XGBoost', dataset_name='Credit Card'
)

### 9.2 Test Set Evaluation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

xgb_cc_results = evaluate_model(
    xgb_cc, X_test_cc, y_test_cc,
    model_name='XGBoost',
    dataset_name='Credit Card',
    ax_cm=axes[0]
)

plot_pr_curves([xgb_cc_results], 'PR Curve — XGBoost (Credit Card)', axes[1])

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'xgb_cc_eval.png'), dpi=150, bbox_inches='tight')
plt.show()

## 10. Model Comparison — Credit Card

In [ ]:
cc_comparison = pd.DataFrame([
    {**{'model': r['model']}, **{k: v for k, v in r.items() if k not in ('model','dataset','pr_curve')}}
    for r in [lr_cc_results, xgb_cc_results]
])

print("=== Credit Card Model Comparison (Test Set) ===")
print(cc_comparison.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_pr_curves([lr_cc_results, xgb_cc_results],
               'Precision-Recall Curves — Credit Card', axes[0])

metrics = ['auc_pr', 'f1', 'precision', 'recall']
x = np.arange(len(metrics))
width = 0.35

bars1 = axes[1].bar(x - width/2, [lr_cc_results[m]  for m in metrics],
                    width, label='Logistic Regression', color='steelblue', edgecolor='white')
bars2 = axes[1].bar(x + width/2, [xgb_cc_results[m] for m in metrics],
                    width, label='XGBoost',             color='tomato',    edgecolor='white')

axes[1].set_xticks(x)
axes[1].set_xticklabels(['AUC-PR', 'F1', 'Precision', 'Recall'])
axes[1].set_title('Metric Comparison — Credit Card', fontsize=11, fontweight='bold')
axes[1].set_ylim(0, 1.1)
axes[1].legend()

for bar in list(bars1) + list(bars2):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

plt.suptitle('Credit Card Fraud — Model Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'cc_model_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

### 10.1 Feature Importance — XGBoost (Credit Card)

In [ ]:
importance_dict_cc = xgb_cc.get_booster().get_score(importance_type='gain')

importance_cc = (
    pd.Series(importance_dict_cc)
    .sort_values(ascending=False)
    .head(15)
)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(importance_cc.index, importance_cc.values,
               color='mediumseagreen', edgecolor='white')
ax.invert_yaxis()
ax.set_xlabel('Feature Importance (Gain)')
ax.set_title('Top 15 Feature Importances — XGBoost (Credit Card)',
             fontsize=12, fontweight='bold')
ax.axvline(importance_cc.values.mean(), color='tomato', linestyle='--',
           linewidth=1.5, label='Mean importance')
ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'xgb_cc_feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()

print("Top 15 features by XGBoost gain (credit card):")
print(importance_cc.to_string())

---
## 11. Master Comparison Table (All Models, Both Datasets)
---

In [ ]:
all_results = []
for res in [lr_fraud_results, xgb_fraud_results, lr_cc_results, xgb_cc_results]:
    row = {k: v for k, v in res.items() if k != 'pr_curve'}
    all_results.append(row)

master_df = pd.DataFrame(all_results)
master_df.columns = ['Model', 'Dataset', 'AUC-PR', 'F1', 'Precision', 'Recall']
master_df = master_df.sort_values(['Dataset', 'AUC-PR'], ascending=[True, False])

print("=" * 70)
print("MASTER MODEL COMPARISON TABLE")
print("=" * 70)
print(master_df.to_string(index=False))
print()
print("PRIMARY METRIC: AUC-PR  (higher is better on imbalanced data)")

In [ ]:
# Cross-validation summary
cv_summary = pd.DataFrame([
    {'Model': 'LR',      'Dataset': 'E-commerce',  **lr_fraud_cv},
    {'Model': 'XGBoost', 'Dataset': 'E-commerce',  **xgb_fraud_cv},
    {'Model': 'LR',      'Dataset': 'Credit Card',  **lr_cc_cv},
    {'Model': 'XGBoost', 'Dataset': 'Credit Card',  **xgb_cc_cv},
])

print("=" * 70)
print("CROSS-VALIDATION SUMMARY (5-Fold, Training Set)")
print("=" * 70)
print(cv_summary.to_string(index=False))

## 12. Model Selection and Justification

### Selected Model: XGBoost (both datasets)

**Quantitative reasoning:**
- XGBoost achieves a higher AUC-PR and F1-Score than Logistic Regression on both datasets
  (exact values visible in the master comparison table above).
- AUC-PR is the primary selection metric because both datasets are severely imbalanced —
  a high AUC-PR means the model maintains high precision even as recall increases,
  which directly translates to fewer false positives at any operating threshold.

**Qualitative reasoning:**

| Factor | Logistic Regression | XGBoost |
|--------|--------------------|---------| 
| Captures non-linear fraud patterns | No | Yes |
| Handles feature interactions | No | Yes |
| Feature importance built-in | No (coefficients only) | Yes (gain, SHAP-compatible) |
| Training time | Fast | Moderate |
| SHAP compatibility | Partial | Full (TreeSHAP) |
| Production deployment | Simple | Moderate complexity |

**Why XGBoost over alternatives (Random Forest / LightGBM)?**
- XGBoost is the most widely validated algorithm in fraud detection literature.
- It natively supports `eval_metric='aucpr'` — optimising directly for our primary metric.
- TreeSHAP integration in Task 3 is fastest and most accurate for tree-based models.

**Business interpretation:**
- XGBoost's ability to capture non-linear relationships is critical here: fraud patterns
  are rarely simple linear thresholds. A fraudster who barely misses one trip-wire will
  trigger other signals that a linear model cannot combine.
- The confusion matrix shows the trade-off between false positives (frustrated legitimate
  customers) and false negatives (missed fraud losses). The operating threshold can be
  adjusted in production to shift this balance based on business priorities.

**Next step:** Task 3 will apply SHAP to the selected XGBoost models to explain individual
predictions and produce actionable business recommendations.

## 13. Save Trained Models

In [ ]:
# Save all four models so Task 3 (SHAP) can load them without retraining.
# pickle serialises the entire fitted model object to a binary file.

models_to_save = {
    'lr_fraud.pkl'  : lr_fraud,
    'xgb_fraud.pkl' : xgb_fraud,
    'lr_cc.pkl'     : lr_cc,
    'xgb_cc.pkl'    : xgb_cc,
}

for filename, model in models_to_save.items():
    path = os.path.join(MODELS_DIR, filename)
    with open(path, 'wb') as f:
        pickle.dump(model, f)
    print(f'  ✓ Saved: {path}')

# Also save feature column names — needed by SHAP to label the axes
feature_cols = {
    'fraud_features': list(X_train_f.columns),
    'cc_features'   : list(X_train_cc.columns),
}
with open(os.path.join(MODELS_DIR, 'feature_cols.pkl'), 'wb') as f:
    pickle.dump(feature_cols, f)

print('\n✓ All models and feature column names saved to models/')

---
## Task 2 Complete — Summary

| Dataset | Model | AUC-PR | F1 | Selected? |
|---------|-------|--------|----|-----------|
| E-commerce | Logistic Regression | *see output above* | *see output above* | No |
| E-commerce | XGBoost | *see output above* | *see output above* | **Yes** |
| Credit Card | Logistic Regression | *see output above* | *see output above* | No |
| Credit Card | XGBoost | *see output above* | *see output above* | **Yes** |

**Files saved:**
```
models/
├── lr_fraud.pkl        ← Logistic Regression (e-commerce)
├── xgb_fraud.pkl       ← XGBoost (e-commerce)  ← SELECTED
├── lr_cc.pkl           ← Logistic Regression (credit card)
├── xgb_cc.pkl          ← XGBoost (credit card) ← SELECTED
└── feature_cols.pkl    ← column name lists for SHAP

notebooks/figures/
├── lr_fraud_eval.png
├── xgb_fraud_eval.png
├── fraud_model_comparison.png
├── xgb_fraud_feature_importance.png
├── lr_cc_eval.png
├── xgb_cc_eval.png
├── cc_model_comparison.png
└── xgb_cc_feature_importance.png
```

**Next: Task 3 — `shap-explainability.ipynb`** loads `xgb_fraud.pkl` and `xgb_cc.pkl`
and generates SHAP Summary Plots, Force Plots, and business recommendations.